# ChakraModel ViT cloud retraining

Select a GPU runtime. This notebook searches mounted datasets, downloads official Kvasir-SEG when absent, validates pairs, retrains the upgraded ViT, and saves checkpoints.

In [ ]:
import os, sys, subprocess, pathlib, shutil
from pathlib import Path
REPO_URL = 'https://github.com/varshasrao58/chakramodel.git'
WORKDIR = Path('/content/chakramodel') if Path('/content').exists() else Path.cwd() / 'chakramodel'
if not (WORKDIR / 'src/chakra_transformer/train_transformer.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.16'], check=True)
print(subprocess.run(['nvidia-smi'], text=True, capture_output=True).stdout[:2000])

## Dataset discovery

The search checks the repository, Google Drive, Kaggle input, `/mnt/data`, and `/workspace`. If no valid image-mask dataset is found, the official Simula Kvasir-SEG downloader is used.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
roots = [WORKDIR, Path('/content/drive/MyDrive'), Path('/kaggle/input'), Path('/mnt/data'), Path('/workspace')]
def score(root):
    image_dir = next((root / x for x in ('images', 'Kvasir-SEG/images', 'kvasir-seg/images') if (root / x).is_dir()), None)
    mask_dir = next((root / x for x in ('masks', 'Kvasir-SEG/masks', 'kvasir-seg/masks') if (root / x).is_dir()), None)
    if not image_dir or not mask_dir: return 0, None, None
    images = {p.stem for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS}
    masks = {p.stem for p in mask_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS}
    return len(images & masks), image_dir, mask_dir
candidates = []
for root in roots:
    if root.exists():
        candidates.append(root)
        candidates.extend(p for p in root.iterdir() if p.is_dir())
best = max((score(p)[0], p) for p in set(candidates)) if candidates else (0, None)
if best[0] == 0:
    print('No valid mounted dataset; downloading official Kvasir-SEG')
    subprocess.run([sys.executable, 'src/download_kvasir.py'], check=True)
else:
    _, source_root = best
    _, image_dir, mask_dir = score(source_root)
    target = WORKDIR / 'data/kvasir-seg'
    target.mkdir(parents=True, exist_ok=True)
    for name, source in [('images', image_dir), ('masks', mask_dir)]:
        link = target / name
        if not link.exists(): link.symlink_to(source, target_is_directory=True)
    print(f'Using {best[0]} pairs from {source_root}')
images_dir, masks_dir = WORKDIR / 'data/kvasir-seg/images', WORKDIR / 'data/kvasir-seg/masks'
image_files = [p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
paired = [p for p in image_files if any((masks_dir / (p.stem + ext)).exists() for ext in IMAGE_EXTS)]
print(f'Images: {len(image_files)}, paired: {len(paired)}')
assert len(paired) >= 100, 'At least 100 valid image-mask pairs are required.'

## Retrain and persist checkpoints

T4 defaults: batch size 2, accumulation 8, AMP enabled. Set environment variables before this cell to change them.

In [ ]:
epochs = os.environ.get('CHAKRA_EPOCHS', '50')
batch = os.environ.get('CHAKRA_BATCH_SIZE', '2')
accum = os.environ.get('CHAKRA_GRAD_ACCUM', '8')
workers = os.environ.get('CHAKRA_WORKERS', '2')
latest = WORKDIR / 'weights/chakra_transformer_latest.pth'
cmd = [sys.executable, 'src/chakra_transformer/train_transformer.py', '--epochs', epochs, '--batch-size', batch, '--grad-accum', accum, '--workers', workers]
if latest.exists(): cmd += ['--resume', str(latest)]
subprocess.run(cmd, check=True)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    backup = Path('/content/drive/MyDrive/chakramodel_weights')
    backup.mkdir(parents=True, exist_ok=True)
    for checkpoint in (WORKDIR / 'weights').glob('chakra_transformer_*.pth'): shutil.copy2(checkpoint, backup / checkpoint.name)
    print('Saved checkpoints to', backup)
else: print('Saved checkpoints to', WORKDIR / 'weights')